# Experiment A — Grid Search Ablation Study

**Reads**: `../../data/processed/4_activity_contributions.csv`
**Writes**: `../../experiments/optimization_results.csv`

> Prerequisites: Run the core pipeline first (`core/notebooks/01–10`).


In [5]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","-r","../../requirements.txt"])
print("Dependencies OK")


Dependencies OK


In [6]:
import pandas as pd, numpy as np, os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import warnings; warnings.filterwarnings("ignore")

PROC="../../data/processed"; EXP="../../experiments"
os.makedirs(EXP,exist_ok=True)
print("Libraries imported.")


Libraries imported.


## 1. Load Data


In [7]:
src = os.path.join(PROC,"4_activity_contributions.csv")
assert os.path.exists(src), f"Run core pipeline first. Missing: {src}"
df = pd.read_csv(src)
print(f"Loaded {len(df)} rows from 4_activity_contributions.csv")


Loaded 3240 rows from 4_activity_contributions.csv


## 2. Define Helper Functions
Breaking the logic down into manageable, readable functions.


In [8]:
# These features have extreme spikes (sparse), so they need special math (log curve)
SPARSE_FEATURES = ["enemiesHit", "damageDone", "timeInCombat", "kills"]

def cap_outliers(X_data, outlier_cap_percentage):
    """
    I apply percentile capping here because my exploratory data analysis showed 
    long-tail distributions (like extremely high distanceTraveled) that would 
    otherwise skew the K-Means boundaries.
    """
    if outlier_cap_percentage >= 100:
        return X_data.copy()
        
    X_capped = X_data.copy()
    for col in X_capped.columns:
        cap_value = X_capped[col].quantile(outlier_cap_percentage / 100.0)
        X_capped[col] = X_capped[col].clip(upper=cap_value)
    return X_capped


In [9]:
def scale_features(X_data, scaling_method, feature_list):
    """
    This squashes all numbers between 0 and 1 so distance doesn't overpower kills.
    I explicitly apply log1p to the heavily skewed combat features before MinMax 
    scaling when using 'minmax_log_sparse'. This prevents extreme outliers from 
    compressing my feature space.
    """
    X_scaled = X_data.copy()
    
    if scaling_method == "minmax_log_sparse":
        for col in feature_list:
            if col in SPARSE_FEATURES:
                X_scaled[col] = np.log1p(X_scaled[col])
        scaler = MinMaxScaler()
        return scaler.fit_transform(X_scaled)
        
    elif scaling_method == "minmax_uniform":
        scaler = MinMaxScaler()
        return scaler.fit_transform(X_scaled)
        
    elif scaling_method == "robust":
        scaler = RobustScaler()
        return scaler.fit_transform(X_scaled)
    
    return X_scaled.values


In [10]:
def evaluate_clustering(X_scaled, k_value):
    """
    I run strict (hard) K-Means clustering here to establish my baseline. 
    This represents the 'traditional' rigid way of classifying players.
    Then, I evaluate my cluster separation using Silhouette/CH and density 
    using Davies-Bouldin scores.
    """
    try:
        kmeans = KMeans(n_clusters=k_value, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        
        # If the AI failed to make enough boxes
        if len(np.unique(cluster_labels)) < k_value:
            return False, 0, 0, 0, 0
            
        # Grade 1: Silhouette (How well separated are the boxes? Higher is better)
        sil_score = silhouette_score(X_scaled, cluster_labels)
        
        # Grade 2: Davies-Bouldin (How dense/tight are the boxes? Lower is better)
        db_score = davies_bouldin_score(X_scaled, cluster_labels)
        
        # Grade 3: Calinski-Harabasz (Ratio of between to within-cluster dispersion)
        ch_score = calinski_harabasz_score(X_scaled, cluster_labels)
        
        # Grade 4: Entropy (Are players evenly distributed among the boxes?)
        cluster_counts = np.bincount(cluster_labels, minlength=k_value).astype(float)
        probabilities = cluster_counts / len(cluster_labels)
        entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
        
        return True, sil_score, db_score, ch_score, entropy
        
    except Exception:
        return False, 0, 0, 0, 0


## 3. Define Grid Search Parameters
I designed this grid search to test 4 K-values x 3 feature sets x 3 outlier caps x 3 normalizations = 108 configurations. This proves mathematically that K=3 is my optimal baseline.


In [11]:
feature_sets = {
    8: ["enemiesHit", "damageDone", "timeInCombat", "kills", "itemsCollected", "pickupAttempts", "distanceTraveled", "timeSprinting"],
    9: ["enemiesHit", "damageDone", "kills", "itemsCollected", "pickupAttempts", "timeNearInteractables", "distanceTraveled", "timeSprinting", "timeOutOfCombat"],
   10: ["enemiesHit", "damageDone", "timeInCombat", "kills", "itemsCollected", "pickupAttempts", "timeNearInteractables", "distanceTraveled", "timeSprinting", "timeOutOfCombat"]
}

k_values = [2, 3, 4, 5]
outlier_caps = [95, 98, 100]
scaling_methods = ["minmax_log_sparse", "minmax_uniform", "robust"]

total_runs = len(k_values) * len(feature_sets) * len(outlier_caps) * len(scaling_methods)
print(f"Total combinations to test: {total_runs}")


Total combinations to test: 108


## 4. Run Grid Search
With all logic abstracted into helper functions, the loop is now incredibly clean.


In [12]:
results = []
runs_completed = 0
print("Starting Grid Search...")

for k in k_values:
    for feature_count, feature_list in feature_sets.items():
        for outlier_cap in outlier_caps:
            for scaling_method in scaling_methods:
                
                # 1. Filter available columns
                available_cols = [c for c in feature_list if c in df.columns]
                X_raw = df[available_cols].fillna(0).copy()
                
                # 2. Cap Outliers
                X_capped = cap_outliers(X_raw, outlier_cap)
                
                # 3. Scale Features
                X_scaled = scale_features(X_capped, scaling_method, feature_list)
                
                # 4. Evaluate K-Means Clustering
                is_valid, sil_score, db_score, ch_score, entropy = evaluate_clustering(X_scaled, k)
                
                # 5. Record Result
                results.append({
                    "k": k,
                    "features": feature_count,
                    "feature_set": str(feature_list),
                    "normalization": scaling_method,
                    "outlier_pct": outlier_cap,
                    "silhouette": sil_score,
                    "db_index": db_score,
                    "ch_score": ch_score,
                    "mean_entropy": entropy,
                    "valid": is_valid
                })
                
                runs_completed += 1
                if runs_completed % 20 == 0:
                    print(f"  {runs_completed} / {total_runs} tested...")

print("Search Complete!")


Starting Grid Search...
  20 / 108 tested...
  40 / 108 tested...
  60 / 108 tested...
  80 / 108 tested...
  100 / 108 tested...
Search Complete!


## 5. Save Results


In [13]:
results_df = pd.DataFrame(results)

# Throw away failed models and sort by the highest Silhouette score
results_df = results_df[results_df["valid"] == True]
results_df = results_df.sort_values(by="silhouette", ascending=False).reset_index(drop=True)

output_path = os.path.join(EXP, "optimization_results.csv")
results_df.to_csv(output_path, index=False)

print(f"Done! {len(results_df)} valid configurations saved.")
print("Top 5 Results:")
print(results_df[["k", "features", "normalization", "outlier_pct", "silhouette", "db_index"]].head(5).to_string(index=False))


Done! 108 valid configurations saved.
Top 5 Results:
 k  features     normalization  outlier_pct  silhouette  db_index
 3         8 minmax_log_sparse          100    0.392779  0.935191
 3        10 minmax_log_sparse          100    0.387575  0.954063
 2         8 minmax_log_sparse           95    0.381666  1.111963
 3         8 minmax_log_sparse           95    0.378314  1.022151
 3        10    minmax_uniform          100    0.376753  0.973733
